# ESM2 기반 효소 변이 효과 예측 미니 프로젝트

**목표 공고**: CJ제일제당 R&D 기술소재사업 BIO AI (효소/단백질 설계), 우대사항 "Protein language model, Generative model 활용 경험"

**구성**
- Phase 1: 공개 DMS 벤치마크(베타락타마제, BLAT_ECOLX)로 ESM2 zero-shot 변이 효과 예측 파이프라인 검증
- Phase 2: 실제 관심 있는 산업효소로 파이프라인 확장 (직접 서열 입력)

이 노트북은 Google Colab에서 실행하는 것을 전제로 작성했습니다 (Runtime → Change runtime type → T4 GPU 권장).


## 0. 환경 설정

In [ ]:
!pip install -q transformers torch scipy pandas matplotlib seaborn tqdm


In [ ]:
import torch
from transformers import AutoTokenizer, EsmForMaskedLM
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("⚠️ GPU가 잡히지 않았습니다. Colab 메뉴 [런타임 > 런타임 유형 변경 > T4 GPU]로 바꾼 뒤 다시 실행하세요.")
    print("   CPU로는 아래 배치 최적화를 적용해도 훨씬 오래 걸립니다.")


## 1. Phase 1 — 벤치마크 데이터 준비

베타락타마제(TEM-1, BLAT_ECOLX) deep mutational scan 데이터 (Ranganathan lab, 2015; Meier et al. 2021 ESM 논문에서 사용한 것과 동일한 파일)를 그대로 사용합니다.
출처: https://github.com/facebookresearch/esm/tree/main/examples/variant-prediction

- `mutant` 컬럼 형식: 야생형잔기+위치+변이잔기 (예: `H24C`)
- `offset_idx = 24` → 서열의 첫 글자가 residue 24에 해당 (즉 서열 인덱스 = 위치 - 24)
- 실험 fitness 컬럼: 여러 항생제 선택 강도 중 `"2500"` (2500 μg/mL 암피실린 선택)이 가장 널리 쓰이는 기준. 논문에서도 이 조건에서 ESM1v가 Spearman ρ ≈ 0.7 수준을 보임 — 우리가 재현하려는 목표 수치입니다.


In [ ]:
DATA_URL = "https://raw.githubusercontent.com/facebookresearch/esm/main/examples/variant-prediction/data/BLAT_ECOLX_Ranganathan2015.csv"
df = pd.read_csv(DATA_URL)

WT_SEQUENCE = (
    "HPETLVKVKDAEDQLGARVGYIELDLNSGKILESFRPEERFPMMSTFKVLLCGAVLSRVDAGQEQLGRRIHYSQNDLVEYSPVTEKHLTDGMTVRELCSAAITMSDNTAANLLLTTIGGPKELTAFLHNMGDHVTRLDRWEPELNEAIPNDERDTTMPAAMATTLRKLLTGELLTLASRQQLIDWMEADKVAGPLLRSALPAGWFIADKSGAGERGSRGIIAALGPDGKPSRIVVIYTTGSQATMDERNRQIAEIGASLIKHW"
)
OFFSET_IDX = 24
FITNESS_COL = "2500"

df = df[["mutant", FITNESS_COL]].dropna().reset_index(drop=True)
print(df.shape)
df.head()


## 2. ESM2 모델 로드 및 masked-marginal 채점 함수

Meier et al. (2021)에서 가장 성능이 좋다고 보고한 **masked-marginal** 방식을 사용합니다.
1. 변이 위치를 `<mask>` 토큰으로 치환
2. 모델이 예측한 그 위치의 log-probability 분포를 얻음
3. `LLR = log P(mutant residue) - log P(wildtype residue)`
4. LLR이 높을수록 "이 변이가 야생형보다 더 자연스럽다(= 덜 해롭다)"는 뜻


In [ ]:
MODEL_NAME = "facebook/esm2_t33_650M_UR50D"  # Colab T4로 충분. 느리면 esm2_t12_35M_UR50D로 축소

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = EsmForMaskedLM.from_pretrained(MODEL_NAME).to(device).eval()


In [ ]:
@torch.no_grad()
def masked_marginal_llr(sequence: str, mutant: str, offset_idx: int) -> float:
    """mutant 예: 'H24C' -> wt='H', pos=24, mut='C'"""
    wt_aa, pos, mut_aa = mutant[0], int(mutant[1:-1]), mutant[-1]
    seq_idx = pos - offset_idx  # 0-indexed position in `sequence`
    assert sequence[seq_idx] == wt_aa, f"{mutant}: sequence has {sequence[seq_idx]}, expected {wt_aa}"

    tokens = tokenizer(sequence, return_tensors="pt").to(device)
    token_idx = seq_idx + 1  # +1 for <cls>
    masked_input = tokens["input_ids"].clone()
    masked_input[0, token_idx] = tokenizer.mask_token_id

    logits = model(input_ids=masked_input, attention_mask=tokens["attention_mask"]).logits
    log_probs = torch.log_softmax(logits[0, token_idx], dim=-1)

    wt_id = tokenizer.convert_tokens_to_ids(wt_aa)
    mut_id = tokenizer.convert_tokens_to_ids(mut_aa)
    return (log_probs[mut_id] - log_probs[wt_id]).item()

# 동작 테스트 (1개 변이)
test_mutant = df.iloc[0]["mutant"]
print(test_mutant, "->", masked_marginal_llr(WT_SEQUENCE, test_mutant, OFFSET_IDX))


## 3. 전체 변이에 대해 점수 계산 (위치 단위 배치 처리로 최적화)

**중요한 최적화**: 같은 위치(position)를 마스킹하는 변이가 여러 개 있습니다 (예: position 24는 H24C, H24D, H24E, ... 19개 변이가 모두 같은 마스킹 위치를 씁니다). 마스킹된 시퀀스는 위치별로 딱 1번만 forward pass 하면 그 위치의 20개 아미노산 확률을 전부 얻을 수 있으므로, **변이 개수(~5,000개)가 아니라 서열 길이(~260개) 만큼만 forward pass하면 됩니다.**

추가로 여러 위치를 하나의 배치로 묶어서(`BATCH_SIZE`) GPU에 한 번에 넣으면 더 빨라집니다.

- 이전 방식(변이 1개당 1 forward pass): 4,996번 → GPU에서도 오래 걸리고, CPU라면 10시간 이상 나올 수 있음
- 최적화 방식(위치 1개당 1 forward pass, 배치 처리): 약 260번 ÷ BATCH_SIZE ≈ 20번 내외의 forward pass → GPU에서 1분 이내, CPU에서도 몇 분 수준

먼저 `device`가 `cuda`로 나오는지 위 셀 출력에서 꼭 확인하세요. `cpu`로 나온다면 런타임을 GPU로 바꾸고 이 지점부터 다시 실행하세요.


In [ ]:
BATCH_SIZE = 16  # GPU 메모리 부족(OOM) 에러 나면 8이나 4로 줄이세요

unique_positions = sorted(df["mutant"].apply(lambda m: int(m[1:-1])).unique().tolist())
print(f"고유 변이 위치 수: {len(unique_positions)}개 (전체 변이 {len(df)}개 -> forward pass {len(unique_positions)}번으로 축소)")

base_tokens = tokenizer(WT_SEQUENCE, return_tensors="pt")
base_input_ids = base_tokens["input_ids"][0]
base_attention_mask = base_tokens["attention_mask"][0]

position_logprobs = {}  # position(1-indexed, 서열 기준) -> 그 위치의 log-probability 벡터(vocab,)

with torch.no_grad():
    for i in tqdm(range(0, len(unique_positions), BATCH_SIZE)):
        batch_positions = unique_positions[i:i + BATCH_SIZE]
        batch_input_ids = base_input_ids.unsqueeze(0).repeat(len(batch_positions), 1).clone()
        batch_attention_mask = base_attention_mask.unsqueeze(0).repeat(len(batch_positions), 1)

        token_idxs = []
        for j, pos in enumerate(batch_positions):
            seq_idx = pos - OFFSET_IDX
            token_idx = seq_idx + 1  # +1 for <cls>
            batch_input_ids[j, token_idx] = tokenizer.mask_token_id
            token_idxs.append(token_idx)

        logits = model(
            input_ids=batch_input_ids.to(device),
            attention_mask=batch_attention_mask.to(device),
        ).logits

        for j, pos in enumerate(batch_positions):
            log_probs = torch.log_softmax(logits[j, token_idxs[j]], dim=-1).cpu()
            position_logprobs[pos] = log_probs

def llr_from_cache(mutant: str) -> float:
    wt_aa, pos, mut_aa = mutant[0], int(mutant[1:-1]), mutant[-1]
    log_probs = position_logprobs[pos]
    wt_id = tokenizer.convert_tokens_to_ids(wt_aa)
    mut_id = tokenizer.convert_tokens_to_ids(mut_aa)
    return (log_probs[mut_id] - log_probs[wt_id]).item()

df["esm2_llr"] = df["mutant"].apply(llr_from_cache)
df = df.dropna(subset=["esm2_llr"])
print(df.shape)


## 4. 검증: 실험값과의 상관관계

In [ ]:
rho, pval = spearmanr(df[FITNESS_COL], df["esm2_llr"])
print(f"Spearman correlation (ESM2 zero-shot vs experimental fitness @ {FITNESS_COL}): rho={rho:.3f}, p={pval:.2e}")
print("참고: Meier et al. 2021 논문의 ESM1v 앙상블 기준 이 조건에서 rho ≈ 0.7 수준으로 보고됨")

plt.figure(figsize=(6,5))
plt.scatter(df["esm2_llr"], df[FITNESS_COL], s=6, alpha=0.4)
plt.xlabel("ESM2 zero-shot LLR (predicted mutation effect)")
plt.ylabel(f"Experimental fitness ({FITNESS_COL})")
plt.title(f"BLAT_ECOLX: predicted vs experimental (Spearman rho={rho:.3f})")
plt.tight_layout()
plt.savefig("phase1_blat_ecolx_correlation.png", dpi=150)
plt.show()


## 5. Phase 2 — 산업효소로 확장: 파이타아제 & 알파-아밀라아제

CJ제일제당 BIO(기술소재)사업과 맞닿는 두 산업효소를 골랐습니다. 둘 다 UniProt reviewed(Swiss-Prot) 항목이라 서열과 활성부위 주석이 신뢰할 수 있는 수준입니다.

| 효소 | UniProt | 유기체 | 산업적 용도 |
|---|---|---|---|
| Phytase A | [P34752](https://www.uniprot.org/uniprotkb/P34752/entry) | *Aspergillus niger* | 사료 첨가 효소 — 곡물 사료 중 인(phosphorus) 이용률을 높여 사료 효율 개선 및 축산 분뇨의 인 배출 저감. CJ BIO의 사료용 아미노산/효소 포트폴리오와 가장 직접적으로 연결됨 |
| Alpha-amylase (amyS) | [P06278](https://www.uniprot.org/uniprotkb/P06278/entry) | *Bacillus licheniformis* | 전분 액화/당화, 식품·세제 산업 표준 효소. 발효 공정 원료(전분당) 생산과 연결됨 |

두 서열 모두 UniProt에 등록된 precursor(신호서열 포함) 형태 그대로 사용합니다 — 활성부위 잔기 번호가 이 번호 체계 그대로 매겨져 있어서 사후 대조가 쉽습니다 (offset_idx=1).


In [ ]:
ENZYMES = {
    "phytase_PHYA_ASPNG_P34752": {
        "sequence": (
            "MGVSAVLLPLYLLSGVTSGLAVPASRNQSSCDTVDQGYQCFSETSHLWGQYAPFFSLANE"
            "SVISPEVPAGCRVTFAQVLSRHGARYPTDSKGKKYSALIEEIQQNATTFDGKYAFLKTYN"
            "YSLGADDLTPFGEQELVNSGIKFYQRYESLTRNIVPFIRSSGSSRVIASGKKFIEGFQST"
            "KLKDPRAQPGQSSPKIDVVISEASSSNNTLDPGTCTVFEDSELADTVEANFTATFVPSIR"
            "QRLENDLSGVTLTDTEVTYLMDMCSFDTISTSTVDTKLSPFCDLFTHDEWINYDYLQSLK"
            "KYYGHGAGNPLGPTQGVGYANELIARLTHSPVHDDTSSNHTLDSSPATFPLNSTLYADFS"
            "HDNGIISILFALGLYNGTKPLSTTTVENITQTDGFSSAWTVPFASRLYVEMMQCQAEQEP"
            "LVRVLVNDRVVPLHGCPVDALGRCTRDSFVRGLSFARSGGDWAECFA"
        ),
        # UniProt ACT_SITE/BINDING 주석 (precursor 기준, signal peptide 1-23 포함)
        "active_site_positions": [82],  # His82, 촉매 nucleophile
        "binding_site_positions": [50, 51, 81, 82, 85, 88, 165, 211, 301, 361, 362],  # phytate 결합 포켓
    },
    "alpha_amylase_AMY_BACLI_P06278": {
        "sequence": (
            "MKQQKRLYARLLTLLFALIFLLPHSAAAAANLNGTLMQYFEWYMPNDGQHWKRLQNDSAY"
            "LAEHGITAVWIPPAYKGTSQADVGYGAYDLYDLGEFHQKGTVRTKYGTKGELQSAIKSLH"
            "SRDINVYGDVVINHKGGADATEDVTAVEVDPADRNRVISGEHRIKAWTHFHFPGRGSTYS"
            "DFKWHWYHFDGTDWDESRKLNRIYKFQGKAWDWEVSNENGNYDYLMYADIDYDHPDVAAE"
            "IKRWGTWYANELQLDGFRLDAVKHIKFSFLRDWVNHVREKTGKEMFTVAEYWQNDLGALE"
            "NYLNKTNFNHSVFDVPLHYQFHAASTQGGGYDMRKLLNSTVVSKHPLKAVTFVDNHDTQP"
            "GQSLESTVQTWFKPLAYAFILTRESGYPQVFYGDMYGTKGDSQREIPALKHKIEPILKAR"
            "KQYAYGAQHDYFDHHDIVGWTREGDSSVANSGLAALITDGPGGAKRMYVGRQNAGETWHD"
            "ITGNRSEPVVINSEGWGEFHVNGGSVSIYVQR"
        ),
        # UniProt ACT_SITE 주석: nucleophile(260), proton donor(290)
        "active_site_positions": [260, 290],
        # Ca2+/Na+ 결합 잔기 (구조 안정성에 중요)
        "binding_site_positions": [133, 190, 210, 212, 223, 229, 231, 233, 264, 329, 331, 357, 435, 436, 459],
    },
}

AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")

def saturation_mutagenesis(sequence: str, batch_size: int = 16) -> pd.DataFrame:
    """Phase 1과 동일한 위치-단위 배치 최적화. 서열 길이만큼만 forward pass."""
    tokens = tokenizer(sequence, return_tensors="pt")
    input_ids = tokens["input_ids"][0]
    attention_mask = tokens["attention_mask"][0]
    positions = list(range(1, len(sequence) + 1))

    pos_logprobs = {}
    with torch.no_grad():
        for i in tqdm(range(0, len(positions), batch_size), leave=False):
            batch_positions = positions[i:i + batch_size]
            batch_input_ids = input_ids.unsqueeze(0).repeat(len(batch_positions), 1).clone()
            batch_attention_mask = attention_mask.unsqueeze(0).repeat(len(batch_positions), 1)
            token_idxs = []
            for j, pos in enumerate(batch_positions):
                token_idx = pos  # offset=1 -> seq_idx=pos-1, token_idx=seq_idx+1=pos
                batch_input_ids[j, token_idx] = tokenizer.mask_token_id
                token_idxs.append(token_idx)
            logits = model(
                input_ids=batch_input_ids.to(device),
                attention_mask=batch_attention_mask.to(device),
            ).logits
            for j, pos in enumerate(batch_positions):
                pos_logprobs[pos] = torch.log_softmax(logits[j, token_idxs[j]], dim=-1).cpu()

    results = []
    for pos in positions:
        wt_aa = sequence[pos - 1]
        wt_id = tokenizer.convert_tokens_to_ids(wt_aa)
        log_probs = pos_logprobs[pos]
        for mut_aa in AMINO_ACIDS:
            if mut_aa == wt_aa:
                continue
            mut_id = tokenizer.convert_tokens_to_ids(mut_aa)
            score = (log_probs[mut_id] - log_probs[wt_id]).item()
            results.append({"mutant": f"{wt_aa}{pos}{mut_aa}", "position": pos, "wt": wt_aa, "mut": mut_aa, "esm2_llr": score})
    return pd.DataFrame(results)

sat_results = {}
for name, info in ENZYMES.items():
    print(f"=== {name} (서열 길이 {len(info['sequence'])}) ===")
    sat_df = saturation_mutagenesis(info["sequence"])
    sat_df.to_csv(f"phase2_{name}_saturation_mutagenesis.csv", index=False)
    sat_results[name] = sat_df
    print(f"저장 완료: phase2_{name}_saturation_mutagenesis.csv ({len(sat_df)}개 변이)\n")


In [ ]:
for name, info in ENZYMES.items():
    sat_df = sat_results[name]
    active = set(info["active_site_positions"])
    binding = set(info["binding_site_positions"]) - active

    active_scores = sat_df[sat_df["position"].isin(active)]["esm2_llr"]
    binding_scores = sat_df[sat_df["position"].isin(binding)]["esm2_llr"]
    other_scores = sat_df[~sat_df["position"].isin(active | binding)]["esm2_llr"]

    print(f"=== {name} : sanity check ===")
    print(f"  촉매 active site 잔기(n={len(active)}) 변이 평균 LLR : {active_scores.mean():.2f}  (낮을수록 모델이 '해롭다'고 예측)")
    print(f"  결합부위 잔기(n={len(binding)}) 변이 평균 LLR       : {binding_scores.mean():.2f}")
    print(f"  나머지 위치 변이 평균 LLR                          : {other_scores.mean():.2f}")
    print("  -> active/binding 잔기의 평균이 나머지보다 뚜렷하게 낮으면, 모델이 알려진 기능 잔기를 올바르게 '중요하다'고 인식하고 있다는 뜻입니다.\n")

    top5 = sat_df.sort_values("esm2_llr", ascending=False).head(5)
    print(f"  상위 5개 후보 변이(활성/안정성 개선 가능성이 높다고 모델이 예측):")
    print(top5[["mutant", "esm2_llr"]].to_string(index=False))
    print()


In [ ]:
import seaborn as sns

fig, axes = plt.subplots(len(ENZYMES), 1, figsize=(14, 4 * len(ENZYMES)))
if len(ENZYMES) == 1:
    axes = [axes]

for ax, (name, sat_df) in zip(axes, sat_results.items()):
    pivot = sat_df.pivot(index="mut", columns="position", values="esm2_llr")
    sns.heatmap(pivot, cmap="RdBu_r", center=0, ax=ax, cbar_kws={"label": "ESM2 LLR"})
    ax.set_title(f"{name} — position x mutant amino acid LLR heatmap")
    ax.set_xlabel("Position")
    ax.set_ylabel("Mutant AA")

plt.tight_layout()
plt.savefig("phase2_heatmaps.png", dpi=150)
plt.show()


## 6. 결과 요약 정리

- Phase 1 결과(rho 값)를 여기 기록:
- Phase 2에서 도출한 상위 후보 변이체 3~5개와 그 위치:
- 활성부위 잔기가 실제로 모델에서 낮은 점수를 받았는지 여부:
- 한계: wet-lab 검증 없는 in-silico 가설 생성 단계임
